# 07 · Climate_TemperaturaMinima · agregado municipio-día

Calcula **Temperatura mínima** por municipio usando la mediana no ponderada de estaciones aceptadas y conserva la cobertura.

In [ ]:
from pathlib import Path
import subprocess
import sys

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/cybercolombia/suelosabio.git'
REPO_REF = 'feature/SCRUM-16'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        remote_ref = f'refs/remotes/origin/{REPO_REF}'
        subprocess.run(['git', 'fetch', '--depth', '1', 'origin', f'+refs/heads/{REPO_REF}:{remote_ref}'], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'checkout', '-B', REPO_REF, remote_ref], cwd=REPO_DIR, check=True)
    PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
else:
    candidatos = [REPO_DIR / 'notebooks' / 'ClimatePipeline', REPO_DIR / 'ClimatePipeline', REPO_DIR]
    PIPELINE_DIR = next((p for p in candidatos if (p / 'DatasetConfig.py').exists()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('No se encontró notebooks/ClimatePipeline.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))


In [ ]:
import json
import time
import pandas as pd
from ClimateProcessingUtils import ahora_proyecto, detectar_commit, escribir_json_atomico, escribir_parquet_atomico, slugificar
from DatasetConfig import cargar_configuracion_datasets
from ScalarMunicipalAggregation import AGGREGATION_VERSION, agregar_escalar_municipal

VARIABLE_NOMBRE = 'temperatura_minima'
DATASET_ID = 'afdg-3zpb'
CONSOLIDACION_ENTRADA = 'cierre_temperatura_minima_2024_2025_v1'
AGREGACION_NOMBRE = 'temperatura_minima_municipio_dia_2024_2025_v1'
FECHA_INICIO = '2024-01-01'
FECHA_FIN = '2025-12-31'
COBERTURA_MINIMA_PCT = 50.0
EJECUTAR_AGREGACION_MUNICIPAL = False
GUARDAR_RESULTADOS = True
SOBRESCRIBIR_RESULTADOS = False

CONFIG=cargar_configuracion_datasets(in_colab=IN_COLAB)
CLIMATE_INPUT_DIR=CONFIG.processed_root/'clima_diario_curado'/f'variable={VARIABLE_NOMBRE}'/f'fuente={DATASET_ID}'/f'consolidacion={slugificar(CONSOLIDACION_ENTRADA)}'
GEOGRAPHY_INPUT_DIR=CONFIG.canonical_geography_root_for(VARIABLE_NOMBRE)
OUTPUT_DIR=CONFIG.processed_root/'clima_municipal'/f'variable={VARIABLE_NOMBRE}'/f'fuente={DATASET_ID}'/f'agregacion={slugificar(AGREGACION_NOMBRE)}'
print({'variable':VARIABLE_NOMBRE,'clima':str(CLIMATE_INPUT_DIR),'geografia':str(GEOGRAPHY_INPUT_DIR),'salida':str(OUTPUT_DIR),'ejecutar':EJECUTAR_AGREGACION_MUNICIPAL})


In [ ]:
if not EJECUTAR_AGREGACION_MUNICIPAL:
    print('Agregación municipal desactivada. Active la bandera después de revisar el plan.')
else:
    inicio_reloj=time.perf_counter(); inicio=ahora_proyecto()
    clima_manifest=json.loads((CLIMATE_INPUT_DIR/'manifest.json').read_text(encoding='utf-8'))
    geo_manifest=json.loads((GEOGRAPHY_INPUT_DIR/'manifest.json').read_text(encoding='utf-8'))
    if clima_manifest.get('estado')!='COMPLETA': raise RuntimeError('La consolidación estación-día no está completa.')
    if not str(geo_manifest.get('estado','')).startswith('COMPLETA'): raise RuntimeError('La geografía no está completa.')
    archivos=sorted(CLIMATE_INPUT_DIR.glob('departamento=*/anio=*/mes=*/observaciones_estacion_dia.parquet'))
    if len(archivos)!=48: raise RuntimeError(f'Se esperaban 48 particiones y existen {len(archivos)}.')
    diario=pd.concat([pd.read_parquet(p) for p in archivos],ignore_index=True)
    estaciones=pd.read_parquet(GEOGRAPHY_INPUT_DIR/'estaciones_municipio.parquet')
    divipola=pd.read_parquet(GEOGRAPHY_INPUT_DIR/'divipola_municipios.parquet')
    resultado=agregar_escalar_municipal(diario,estaciones,divipola,FECHA_INICIO,FECHA_FIN,cobertura_minima_pct=COBERTURA_MINIMA_PCT)
    if GUARDAR_RESULTADOS:
        tabla=resultado.diario_municipal.assign(anio=lambda x:x.fecha.dt.year,mes=lambda x:x.fecha.dt.month)
        particiones=[]
        for (departamento,anio,mes),bloque in tabla.groupby(['departamento','anio','mes'],sort=True):
            ruta=OUTPUT_DIR/f'departamento={departamento}'/f'anio={int(anio)}'/f'mes={int(mes):02d}'/'valor_municipio_dia.parquet'
            escribir_parquet_atomico(bloque.drop(columns=['anio','mes']),ruta,sobrescribir=SOBRESCRIBIR_RESULTADOS)
            particiones.append({'ruta':str(ruta),'filas':len(bloque)})
        escribir_parquet_atomico(resultado.resumen_municipio,OUTPUT_DIR/'resumen_municipios.parquet',sobrescribir=SOBRESCRIBIR_RESULTADOS)
        fin=ahora_proyecto()
        escribir_json_atomico({'estado':'COMPLETA','aggregation_version':AGGREGATION_VERSION,'variable':VARIABLE_NOMBRE,'dataset_id':DATASET_ID,'commit':detectar_commit(REPO_DIR),'inicio':inicio.isoformat(),'fin':fin.isoformat(),'duracion_segundos':round(time.perf_counter()-inicio_reloj,2),'metricas':resultado.metricas,'particiones':particiones,'entrada_clima':str(CLIMATE_INPUT_DIR),'entrada_geografia':str(GEOGRAPHY_INPUT_DIR)},OUTPUT_DIR/'manifest.json',sobrescribir=True)
        print(f'Agregación municipal guardada en {OUTPUT_DIR}')
    display(resultado.resumen_municipio)
